# Transform Geolocation Data

1. filter invalid rows i.e rows with geolocation_zip_code_prefix as null and drop duplicates
2. Standardize geolocation_city to lowercase and geolocation_state to uppercase
3. write transformed data to silver table

In [0]:
#Imports
from pyspark.sql.functions import col, trim, lower, upper

In [0]:
geolocation_df = spark.read.table("olist_catalog.bronze.geolocation")

### Step1 - filter invalid rows i.e rows with geolocation_zip_code_prefix as null and drop duplicates

In [0]:
geolocation_valid_df = (
    geolocation_df.filter(col("geolocation_zip_code_prefix").isNotNull())
        .dropDuplicates()
)

### Step2 - Standardize geolocation_city to lowercase and geolocation_state to uppercase

In [0]:
geolocation_final_df = geolocation_valid_df.withColumns({'geolocation_city':lower(trim(col("geolocation_city"))),'geolocation_state':upper(trim(col("geolocation_state")))})

### Step3 - write transformed data to silver table

In [0]:
(
    geolocation_final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.silver.geolocation")
)

In [0]:
%sql
select * from olist_catalog.silver.geolocation